# 03 - Train Reactant/Class Model

QLoRA fine-tune for compact `product_smiles -> reaction_class + reactants` prediction.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub trl safetensors sentencepiece rdkit

import gc
import getpass
import json
import os
import random
from pathlib import Path

import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login, whoami
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
HF_MODEL_REPO_NAME = 'retro-reactants-qwen25-7b'
WORK_DIR = Path('/content/retro_condition_agent')
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'trainer_reactants_output'
ADAPTER_DIR = WORK_DIR / 'retro_reactants_lora'
TRAIN_JSONL = DATA_DIR / 'reactants_train.jsonl'
EVAL_JSONL = DATA_DIR / 'reactants_eval.jsonl'

MAX_SEQ_LENGTH = 1024
MAX_STEPS = 3000
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
SEED = 3407
USE_BF16 = True

random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSONL.exists(), f'Missing {TRAIN_JSONL}. Run 01_prepare_reactant_data.ipynb first.'
assert EVAL_JSONL.exists(), f'Missing {EVAL_JSONL}. Run 01_prepare_reactant_data.ipynb first.'
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste Hugging Face token with write access: ')

login(token=HF_TOKEN)
HF_USERNAME = whoami(token=HF_TOKEN)['name']
LORA_REPO_ID = f'{HF_USERNAME}/{HF_MODEL_REPO_NAME}-lora'
print('LoRA repo:', LORA_REPO_ID)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

raw_dataset = load_dataset('json', data_files={'train': str(TRAIN_JSONL), 'eval': str(EVAL_JSONL)})

def render_text(example):
    messages = example['messages']
    full = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    if tokenizer.eos_token and not full.endswith(tokenizer.eos_token):
        full += tokenizer.eos_token
    return {'full_text': full, 'prompt_text': prompt}

def tokenize_with_assistant_mask(example):
    full = tokenizer(example['full_text'], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    prompt = tokenizer(example['prompt_text'], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    labels = list(full['input_ids'])
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full['labels'] = labels
    return full

rendered = raw_dataset.map(render_text, remove_columns=raw_dataset['train'].column_names)
train_dataset = rendered['train'].map(tokenize_with_assistant_mask, remove_columns=rendered['train'].column_names)
eval_dataset = rendered['eval'].map(tokenize_with_assistant_mask, remove_columns=rendered['eval'].column_names)
print(train_dataset)
print('example tokens:', len(train_dataset[0]['input_ids']))


In [ ]:
gc.collect()
torch.cuda.empty_cache()
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16, trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
peft_config = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
def collate(features):
    labels = [f.pop('labels') for f in features]
    batch = tokenizer.pad(features, padding=True, return_tensors='pt')
    max_len = batch['input_ids'].shape[1]
    padded_labels = []
    for label in labels:
        padded_labels.append(label + [-100] * (max_len - len(label)))
    batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return batch

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    warmup_steps=max(1, int(MAX_STEPS * 0.05)),
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    bf16=USE_BF16,
    fp16=False,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    save_total_limit=3,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collate, processing_class=tokenizer)
trainer.train()
print(trainer.evaluate())


In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
(ADAPTER_DIR / 'README.md').write_text(f'''---\nlicense: apache-2.0\nbase_model: {BASE_MODEL}\ntags:\n- chemistry\n- retrosynthesis\n- qlora\n- lora\n---\n\n# {HF_MODEL_REPO_NAME} LoRA\n\nPredicts compact JSON reactants and reaction class from product SMILES.\n''')
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=LORA_REPO_ID, repo_type='model', exist_ok=True)
api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=LORA_REPO_ID, repo_type='model')
print('Uploaded:', LORA_REPO_ID)
